# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
from sklearn.base import clone

configs = [
    {
        'name': 'anchor_word13_char35_C2',
        'word_ngram': (1, 3), 'char_ngram': (3, 5), 'char_analyzer': 'char',
        'word_max_features': 220000, 'char_max_features': 220000,
        'min_df': 2, 'C': 2.0,
    },
    {
        'name': 'charwb_word13_char35_C2',
        'word_ngram': (1, 3), 'char_ngram': (3, 5), 'char_analyzer': 'char_wb',
        'word_max_features': 220000, 'char_max_features': 220000,
        'min_df': 2, 'C': 2.0,
    },
    {
        'name': 'word14_char35_C2',
        'word_ngram': (1, 4), 'char_ngram': (3, 5), 'char_analyzer': 'char',
        'word_max_features': 260000, 'char_max_features': 220000,
        'min_df': 2, 'C': 2.0,
    },
    {
        'name': 'word13_char35_C4',
        'word_ngram': (1, 3), 'char_ngram': (3, 5), 'char_analyzer': 'char',
        'word_max_features': 220000, 'char_max_features': 220000,
        'min_df': 2, 'C': 4.0,
    },
    {
        'name': 'word13_char26_C1',
        'word_ngram': (1, 3), 'char_ngram': (2, 6), 'char_analyzer': 'char',
        'word_max_features': 220000, 'char_max_features': 260000,
        'min_df': 2, 'C': 1.0,
    },
]

def make_model(cfg):
    features = FeatureUnion([
        ('word', TfidfVectorizer(
            analyzer='word', ngram_range=cfg['word_ngram'],
            min_df=cfg['min_df'], sublinear_tf=True,
            max_features=cfg['word_max_features'])),
        ('char', TfidfVectorizer(
            analyzer=cfg['char_analyzer'], ngram_range=cfg['char_ngram'],
            min_df=cfg['min_df'], sublinear_tf=True,
            max_features=cfg['char_max_features'])),
    ])
    return Pipeline([
        ('features', features),
        ('classifier', LinearSVC(C=cfg['C'], class_weight='balanced')),
    ])

results = []
for cfg in configs:
    started = time.time()
    candidate = make_model(cfg)
    candidate.fit(train_part['text'], train_part['model'])
    pred = candidate.predict(valid_part['text'])
    result = {
        'experiment': cfg['name'],
        'accuracy': float(accuracy_score(valid_part['model'], pred)),
        'balanced_accuracy': float(balanced_accuracy_score(valid_part['model'], pred)),
        'training_seconds': round(time.time() - started, 2),
        **{k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()},
    }
    results.append(result)
    print(f"{cfg['name']}: accuracy={result['accuracy']:.4f}, balanced_accuracy={result['balanced_accuracy']:.4f}, seconds={result['training_seconds']}")

results.sort(key=lambda r: (r['balanced_accuracy'], r['accuracy']), reverse=True)
best = results[0]
print('\nRANKED RESULTS:')
for rank, result in enumerate(results, 1):
    print(f"{rank}. {result['experiment']}: balanced_accuracy={result['balanced_accuracy']:.4f}")
print('\nBEST CONFIG:', json.dumps(best, indent=2))

summary = {
    'experiment': 'LLM GMC TF-IDF controlled benchmark',
    'competition': 'name-that-llm',
    'split_method': split_method,
    'train_rows': int(len(train_part)),
    'validation_rows': int(len(valid_part)),
    'total_rows': int(len(train)),
    'model_classes': int(train['model'].nunique()),
    'baseline_balanced_accuracy': 0.28703703703703703,
    'round': 2,
    'best': best,
    'results': results,
}
Path('/kaggle/working/experiment_results.json').write_text(json.dumps(summary, indent=2))
print('Saved benchmark results:', '/kaggle/working/experiment_results.json')


## Result interpretation

The notebook benchmarks multiple configurations on the same prompt-grouped validation split. The reported score is local validation, not the Kaggle leaderboard score. The best configuration is selected only for comparison; no leaderboard submission is made in this experiment.
